# 04 - Delta Live Tables (DLT) Bronze SCADA Pipeline + File Audit Log
**Attach this notebook to a Databricks Delta Live Tables (Lakeflow Declarative) Pipeline.**

### Tables Created in `aemo.bronze`:
1. **`bronze_scada` (DLT Streaming Table):**
   * Streams clean `.csv` files from `/Volumes/aemo/bronze/landing/Dispatch_SCADA/YYYY-MM-DD/*.csv` via Auto Loader (`cloudFiles`).
   * Drops non-data rows (`@dlt.expect_or_drop("is_data_row", "I = 'D'")`).
   * Deduplicates and merges on `(SETTLEMENTDATE, DUID)` via `dlt.apply_changes`.
2. **`dlt_processed_file_log` (DLT Audit Table in `bronze` schema):**
   * Tracks every `source_file` processed by DLT, the exact timestamp it was processed (`first_processed_at_utc` / `last_processed_at_utc`), the number of rows ingested (`dlt_ingested_row_count`), and its `SETTLEMENTDATE` range.

In [0]:
import dlt
from pyspark.sql import functions as F

CATALOG = "aemo"
SCHEMA = "bronze"

LANDING_CSV_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/landing/Dispatch_SCADA"
SCHEMA_LOCATION_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/landing/_dlt_schemas/bronze_scada"

# -------------------------------------------------------------------------
# 1. DLT Streaming View: Auto Loader (`cloudFiles`) over Extracted CSVs
# -------------------------------------------------------------------------
@dlt.view(
    name="scada_raw_csv_stream",
    comment="Incrementally reads extracted AEMO Dispatch_SCADA CSV files from Unity Catalog Volume using Auto Loader.",
)
@dlt.expect_or_drop("is_data_row", "I = 'D'")
@dlt.expect_or_drop("valid_settlementdate", "SETTLEMENTDATE IS NOT NULL")
@dlt.expect_or_drop("valid_duid", "DUID IS NOT NULL")
def scada_raw_csv_stream():
    df_raw = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION_PATH)
        .option("cloudFiles.inferColumnTypes", "false")
        .option("header", "true")
        .option("pathGlobFilter", "*.csv")
        .option("recursiveFileLookup", "true")
        .load(LANDING_CSV_PATH)
    )

    return (
        df_raw
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withColumn("source_file_path", F.col("_metadata.file_path"))
        .withColumn("source_file_size_bytes", F.col("_metadata.file_size"))
        .withColumn("source_file_modification_time", F.col("_metadata.file_modification_time").cast("string"))
        .withColumn("ingested_at_utc", F.current_timestamp().cast("string"))
        .withColumn(
            "settlement_date_part",
            F.regexp_replace(F.substring(F.col("SETTLEMENTDATE"), 1, 10), "/", "-"),
        )
    )

# -------------------------------------------------------------------------
# 2. Target DLT Streaming Table: `bronze_scada` (SCD Type 1 Auto-MERGE)
# -------------------------------------------------------------------------
dlt.create_streaming_table(
    name="bronze_scada",
    comment="Deduplicated Bronze AEMO 5-minute Dispatch SCADA table merged on (SETTLEMENTDATE, DUID).",
    partition_cols=["settlement_date_part"],
)

dlt.apply_changes(
    target="bronze_scada",
    source="scada_raw_csv_stream",
    keys=["SETTLEMENTDATE", "DUID"],
    sequence_by=F.col("ingested_at_utc"),
    except_column_list=["source_file_path", "source_file_size_bytes", "source_file_modification_time"],
    stored_as_scd_type=1,
)

# -------------------------------------------------------------------------
# 3. Target DLT Table: `dlt_processed_file_log` (in `aemo.bronze` schema)
#    Tracks what files were processed by DLT and at what time point
# -------------------------------------------------------------------------
@dlt.table(
    name="dlt_processed_file_log",
    comment="Audit log of all CSV files processed by the DLT pipeline, including processed timestamps and row counts.",
)
def dlt_processed_file_log():
    return (
        dlt.read("bronze_scada")
        .groupBy("source_file", "settlement_date_part")
        .agg(
            F.min("ingested_at_utc").alias("first_processed_at_utc"),
            F.max("ingested_at_utc").alias("last_processed_at_utc"),
            F.count("*").alias("dlt_ingested_row_count"),
            F.min("SETTLEMENTDATE").alias("min_settlement_date"),
            F.max("SETTLEMENTDATE").alias("max_settlement_date"),
        )
    )